# Decision Tree Optimization

In [1]:
import pandas as pd
import json

In [2]:
df = pd.DataFrame({'studied': [1,2,3,4,5,6,7,8,9,10,11,12], 
        'passed': [0,0,0,0,1,1,1,1,1,1,1,1]})

In [3]:
df

,studied,passed
0,1,0
1,2,0
2,3,0
3,4,0
4,5,1
5,6,1
6,7,1
7,8,1
8,9,1
9,10,1


## Mid point

In [4]:
# compute mid points between values of studied
mid_points = []
for i in range(len(df['studied']) - 1):
    mid_point = (df['studied'][i] + df['studied'][i+1]) / 2
    # [i] refer to the fist value and [i+1] refer to the second value and so on
    print(mid_point)
    mid_points.append(mid_point)

1.5
2.5
3.5
4.5
5.5
6.5
7.5
8.5
9.5
10.5
11.5


In [5]:
x = df['studied'].values
x

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12])

In [6]:
print("from 1st item to 2nd last:")
print(x[:-1])
print("from 2nd item to last:")
print(x[1:])

from 1st item to 2nd last:
[ 1  2  3  4  5  6  7  8  9 10 11]
from 2nd item to last:
[ 2  3  4  5  6  7  8  9 10 11 12]


In [7]:
mid_points = (x[:-1] + x[1:]) / 2
print(mid_points)

[ 1.5  2.5  3.5  4.5  5.5  6.5  7.5  8.5  9.5 10.5 11.5]


## Sliding Windows

### Gini Formula

In [8]:
def gini(y):
    if len(y) == 0:
        return 0
    p0 = (y == 0).sum() / len(y)
    p1 = (y == 1).sum() / len(y)
    gini = 1 - (p0**2 + p1**2)
    return gini

In [9]:
# test gini
gini(df['passed'].values)

np.float64(0.4444444444444444)

In [10]:
def weighted_gini(y_left, y_right):
    n = len(y_left) + len(y_right)
    gini_left = gini(y_left)
    gini_right = gini(y_right)
    weighted_gini = (len(y_left) / n) * gini_left + (len(y_right) / n) * gini_right
    return weighted_gini

In [11]:
# Split base on first mid point
mid_point = mid_points[0]
left_node = df[df['studied'] <= mid_point]
right_node = df[df['studied'] > mid_point]
weighted_gini_first = weighted_gini(left_node['passed'].values, right_node['passed'].values)

In [12]:
print(mid_point)
print(left_node)
print(right_node)

1.5
   studied  passed
0        1       0
    studied  passed
1         2       0
2         3       0
3         4       0
4         5       1
5         6       1
6         7       1
7         8       1
8         9       1
9        10       1
10       11       1
11       12       1


In [13]:
# second mid point split
mid_point = mid_points[1]
left_node = df[df['studied'] <= mid_point]
right_node = df[df['studied'] > mid_point]
print(mid_point)
print(left_node)
print(right_node)
weighted_gini_second = weighted_gini(left_node['passed'].values, right_node['passed'].values)


2.5
   studied  passed
0        1       0
1        2       0
    studied  passed
2         3       0
3         4       0
4         5       1
5         6       1
6         7       1
7         8       1
8         9       1
9        10       1
10       11       1
11       12       1


In [14]:
print("Weighted Gini for first split:", weighted_gini_first)
print("Weighted Gini for second split:", weighted_gini_second)

Weighted Gini for first split: 0.36363636363636365
Weighted Gini for second split: 0.26666666666666655


## Sliding Windows

Pseudo code:

1. Sort feature
2. Initialize LEFT counts to zero
3. Initialize RIGHT counts to total class counts
4. Move one sample at a time from RIGHT to LEFT
5. Update counts
6. Compute weighted Gini
7. Track best boundary
8. Convert best boundary to midpoint threshold




In [15]:
import numpy as np

In [16]:
def gini(no_negative, no_positive):
    total = no_negative + no_positive
    if total == 0:
        return 0

    p0 = no_negative/total
    p1 = no_positive/total
    gini = 1 - (p0**2 + p1**2)
    return gini

In [17]:
def best_gini_search(df):

    best_gini = float('inf')
    best_split_value = None

    # Sort features
    sorted_df = df.sort_values(by='studied')
    x = sorted_df['studied'].values
    y = sorted_df['passed'].values

    # initialized count
    left_p0_count = 0
    left_p1_count = 0
    right_p0_count = (y==0).sum()
    right_p1_count = (y==1).sum()

    # print(left_p0_count)
    # print(left_p1_count)
    # print(right_p0_count)
    # print(right_p1_count)

    # initializa total number of data
    n = len(y)
    n_right = n
    n_left = 0

    for i in range(n-1):
        if y[i] == 0:
            left_p0_count += 1
            right_p0_count -= 1
            n_left += 1
            n_right -= 1
        else:
            left_p1_count += 1
            right_p1_count -= 1
            n_left += 1
            n_right -= 1

        print('left false',left_p0_count)
        print('left, true',left_p1_count)
        print('total left',n_left)
        print('right false',right_p0_count)
        print('right, true',right_p1_count)
        print('total right',n_right)
        
        left_gini = gini(left_p0_count, left_p1_count)
        right_gini = gini(right_p0_count, right_p1_count)
        weighted_gini = (n_left / n) *  left_gini + (n_right / n) * right_gini
        print('weighted gini',weighted_gini)
 
        if weighted_gini < best_gini:
            best_gini = weighted_gini
            best_split_value = (x[i] + x[i+1])/2
            #print(best_split_value)
            print('best gini', best_gini)
        
    return best_split_value, best_gini


In [18]:
best_gini_search(df)

left false 1
left, true 0
total left 1
right false 3
right, true 8
total right 11
weighted gini 0.36363636363636365
best gini 0.36363636363636365
left false 2
left, true 0
total left 2
right false 2
right, true 8
total right 10
weighted gini 0.26666666666666655
best gini 0.26666666666666655
left false 3
left, true 0
total left 3
right false 1
right, true 8
total right 9
weighted gini 0.14814814814814814
best gini 0.14814814814814814
left false 4
left, true 0
total left 4
right false 0
right, true 8
total right 8
weighted gini 0.0
best gini 0.0
left false 4
left, true 1
total left 5
right false 0
right, true 7
total right 7
weighted gini 0.13333333333333328
left false 4
left, true 2
total left 6
right false 0
right, true 6
total right 6
weighted gini 0.2222222222222222
left false 4
left, true 3
total left 7
right false 0
right, true 5
total right 5
weighted gini 0.2857142857142858
left false 4
left, true 4
total left 8
right false 0
right, true 4
total right 4
weighted gini 0.3333333333

(np.float64(4.5), np.float64(0.0))

## CUMSUM

In [19]:
y = df['passed'].values
x = df['studied'].values

In [20]:
left_negative = (y == 0).cumsum()
left_positive = (y == 1).cumsum()
print('Left Negative', left_negative)
print('Left Positive', left_positive)

Left Negative [1 2 3 4 4 4 4 4 4 4 4 4]
Left Positive [0 0 0 0 1 2 3 4 5 6 7 8]


In [21]:
negative_total = left_negative[-1]
positive_total = left_positive[-1]
print('Negative Total', negative_total)
print('Positive Total', positive_total)

Negative Total 4
Positive Total 8


In [22]:
right_negative = negative_total - left_negative
right_positive = positive_total - left_positive
print('Right Negative', right_negative)
print('Right Positive', right_positive)

Right Negative [3 2 1 0 0 0 0 0 0 0 0 0]
Right Positive [8 8 8 8 7 6 5 4 3 2 1 0]


In [23]:
def best_gini_cumsum(df):

    best_gini = float('inf')
    best_split_value = None

    # Sort features
    sorted_df = df.sort_values(by='studied')
    x = sorted_df['studied'].values
    y = sorted_df['passed'].values

    # initialized count
    left_negative = (y == 0).cumsum()
    left_positive = (y == 1).cumsum()
    negative_total = left_negative[-1]
    positive_total = left_positive[-1]
    right_negative = negative_total - left_negative
    right_positive = positive_total - left_positive


    # initialize total number of data
    n = len(y)
    n_right = n
    n_left = 0

    for i in range(n-1):
        n_left += 1
        n_right -= 1

        left_negative[i]
        
        left_gini = gini(left_negative[i], left_positive[i])
        right_gini = gini(right_negative[i], right_positive[i])
        weighted_gini = (n_left / n) *  left_gini + (n_right / n) * right_gini
        print('weighted gini',weighted_gini)
 
        if weighted_gini < best_gini:
            best_gini = weighted_gini
            best_split_value = (x[i] + x[i+1])/2
            #print(best_split_value)
            print('best gini', best_gini)
        
    return best_split_value, best_gini


In [24]:
df

,studied,passed
0,1,0
1,2,0
2,3,0
3,4,0
4,5,1
5,6,1
6,7,1
7,8,1
8,9,1
9,10,1


In [27]:
split, best_gini = best_gini_cumsum(df)
print(f"Best gini {best_gini} and the threshold is {split}")

weighted gini 0.36363636363636365
best gini 0.36363636363636365
weighted gini 0.26666666666666655
best gini 0.26666666666666655
weighted gini 0.14814814814814814
best gini 0.14814814814814814
weighted gini 0.0
best gini 0.0
weighted gini 0.13333333333333328
weighted gini 0.2222222222222222
weighted gini 0.2857142857142858
weighted gini 0.3333333333333333
weighted gini 0.37037037037037035
weighted gini 0.4
weighted gini 0.42424242424242425
Best gini 0.0 and the threshold is 4.5


## Vectorization

In [28]:
left_negative = (y == 0).cumsum()
left_positive = (y == 1).cumsum()
print('Left Negative', left_negative)
print('Left Positive', left_positive)

Left Negative [1 2 3 4 4 4 4 4 4 4 4 4]
Left Positive [0 0 0 0 1 2 3 4 5 6 7 8]


In [46]:
left_negative

array([1, 2, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4])

In [47]:
left_positive

array([0, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8])

In [38]:
left_count = left_negative + left_positive
left_count

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12])

In [29]:
negative_total = left_negative[-1]
positive_total = left_positive[-1]
print('Negative Total', negative_total)
print('Positive Total', positive_total)

Negative Total 4
Positive Total 8


In [36]:
n = negative_total + positive_total
n

np.int64(12)

In [33]:
right_negative = negative_total - left_negative
right_positive = positive_total - left_positive
print('Right Negative', right_negative)
print('Right Positive', right_positive)

Right Negative [3 2 1 0 0 0 0 0 0 0 0 0]
Right Positive [8 8 8 8 7 6 5 4 3 2 1 0]


In [39]:
right_count = right_negative + right_positive
right_count

array([11, 10,  9,  8,  7,  6,  5,  4,  3,  2,  1,  0])

In [59]:
def gini(no_negative: np.array, no_positive: np.array, no_total: np.array):
    p0 = np.divide(no_negative, no_total, out=np.zeros_like(no_total, dtype=float), where=no_total!=0)
    p1 = np.divide(no_positive, no_total, out=np.zeros_like(no_total, dtype=float), where=no_total!=0)
    gini = 1 - (p0**2 + p1**2)
    return gini

In [62]:
left_gini = gini(left_negative, left_positive, left_count)
right_gini = gini(right_negative, right_positive, right_count)
weighted_gini = (left_count / n) *  left_gini + (right_count / n) * right_gini

In [63]:
weighted_gini

array([0.36363636, 0.26666667, 0.14814815, 0.        , 0.13333333,
       0.22222222, 0.28571429, 0.33333333, 0.37037037, 0.4       ,
       0.42424242, 0.44444444])

In [65]:
dashboard = np.c_[left_negative,left_positive,left_count,right_negative, right_positive, right_count,left_gini,right_gini, weighted_gini]

In [66]:
dashboard_df = pd.DataFrame(dashboard, columns=['left_negative','left_positive','left_count','right_negative', 'right_positive', 'right_count','left_gini','right_gini', 'weighted_gini'])

In [67]:
dashboard_df

,left_negative,left_positive,left_count,right_negative,right_positive,right_count,left_gini,right_gini,weighted_gini
0,1.0,0.0,1.0,3.0,8.0,11.0,0.000000,0.396694,0.363636
1,2.0,0.0,2.0,2.0,8.0,10.0,0.000000,0.320000,0.266667
2,3.0,0.0,3.0,1.0,8.0,9.0,0.000000,0.197531,0.148148
3,4.0,0.0,4.0,0.0,8.0,8.0,0.000000,0.000000,0.000000
4,4.0,1.0,5.0,0.0,7.0,7.0,0.320000,0.000000,0.133333
5,4.0,2.0,6.0,0.0,6.0,6.0,0.444444,0.000000,0.222222
6,4.0,3.0,7.0,0.0,5.0,5.0,0.489796,0.000000,0.285714
7,4.0,4.0,8.0,0.0,4.0,4.0,0.500000,0.000000,0.333333
8,4.0,5.0,9.0,0.0,3.0,3.0,0.493827,0.000000,0.370370
9,4.0,6.0,10.0,0.0,2.0,2.0,0.480000,0.000000,0.400000


In [74]:
index = dashboard_df['weighted_gini'].argmin()

In [77]:
x[index]

np.int64(4)

In [78]:
threshold = (x[index] + x[index+1])/2
threshold

np.float64(4.5)